# 09 — Estimate and export the thesis models

Run 08 first. One canonical resolution folder supplies the local model rows. Seven
aggregate files are written to RESULTS/100m/final/; an existing result folder is moved
to ARCHIVE before a new run. No input or notebook content hashes are calculated.
The main model families, H1–H5 contrasts and validated Cox covariance remain intact.
Prespecified sensitivities remain separate and their failures are visible. Main results
are saved before optional Cox sensitivities. FIG reads these files without refitting.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import warnings
import shutil
from time import perf_counter

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy
from scipy import stats
from scipy.special import digamma
import statsmodels
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.duration.hazard_regression import PHReg

CONFIG = {
    "cell_size": 100, "run_name": "final",
    "accessibility_sensitivity_minutes": 30, "centre_sensitivity": True,
    "population_transition_sensitivity": True,
    "ph_sensitivity": True, "maxiter_cox_sensitivity": 30,
    "spatial_block_m": 5000,
    "maxiter_poisson": 300, "maxiter_nb2": 300, "maxiter_cox": 300,
    "optimizer_nb2": "lbfgs", "alpha_lower": 1e-6, "alpha_upper": 1e4,
}
START_YEAR, END_YEAR, LAG_YEAR = 2016, 2025, 2015
PROJECT_DIR = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "ANAL").is_dir()), None)
if PROJECT_DIR is None:
    raise RuntimeError("Open this notebook inside the project.")
CELL_SIZE = CONFIG["cell_size"]
if CELL_SIZE < 100 or CELL_SIZE % 100:
    raise ValueError("Use 100 m or a larger multiple of 100 m.")
RESOLUTION = f"{CELL_SIZE}m"
DATA_DIR = PROJECT_DIR / "ANAL" / "data" / RESOLUTION
FEATURE_DIR = DATA_DIR / "routing" / "features"
PANEL_PATH = DATA_DIR / f"raster_quarter_panel_{RESOLUTION}.parquet"
FIRMS_PATH = DATA_DIR / f"firms_assigned_{RESOLUTION}.geoparquet"
RASTER_PATH = DATA_DIR / f"raster_{RESOLUTION}_styria.geoparquet"
H3_BIRTHS_PATH = DATA_DIR / f"births_by_fachgruppe_{RESOLUTION}.parquet"
run_name = CONFIG["run_name"]
if not run_name or Path(run_name).name != run_name or run_name in {".", ".."}:
    raise ValueError("run_name must be a single folder name.")
RESULT_ROOT = (PROJECT_DIR / "RESULTS").resolve()
BUNDLE_DIR = (RESULT_ROOT / RESOLUTION / run_name).resolve()
if not BUNDLE_DIR.is_relative_to(RESULT_ROOT):
    raise ValueError("Result folder must remain below RESULTS.")
if BUNDLE_DIR.exists() and any(BUNDLE_DIR.iterdir()):
    archive = PROJECT_DIR / "ARCHIVE" / "results" / RESOLUTION / (run_name+"_"+datetime.now().strftime("%Y-%m-%d_%H%M%S_%f"))
    if not archive.resolve().is_relative_to((PROJECT_DIR / "ARCHIVE").resolve()):
        raise ValueError("Archive folder must remain below ARCHIVE.")
    archive.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(BUNDLE_DIR), str(archive))
    print("Previous results archived:", archive)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
sample_rows, diagnostic_rows = [], []
coefficient_frames, hypothesis_frames, covariance_frames, plot_frames = {}, {}, {}, {}
fit_summaries, coverage_frames = {}, {}
FOUNDING_FREQUENCY = "quarter"
manifest = dict(status="running", pipeline_version="09-simple-1", resolution_m=CELL_SIZE,
    run_name=run_name, created_at=datetime.now(timezone.utc).isoformat(), config=CONFIG,
    study_period=dict(start_year=START_YEAR, end_year=END_YEAR, lead_in_year=LAG_YEAR),
    time_aggregation=dict(founding="quarter", survival="quarter"),
    software=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                  scipy=scipy.__version__, statsmodels=statsmodels.__version__))
(BUNDLE_DIR / "laufinfo.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"{RESOLUTION}: 2016Q1–2025Q4. Aggregate results: {BUNDLE_DIR}")


## 1. Prepare the two model samples

In [ ]:
def note(model, check, status, detail):
    diagnostic_rows[:] = [r for r in diagnostic_rows if (r.get("model"), r.get("check")) != (model, check)]
    diagnostic_rows.append(dict(model=model, check=check, status=status, detail=detail))

def columns(path):
    return set(pq.ParquetFile(path).schema_arrow.names)

def grid_column(path):
    if "grid_id" not in columns(path):
        raise ValueError(f"{path.name}: missing canonical grid_id; run 08 in the new resolution folder.")
    return "grid_id"


def require_columns(path, required):
    available = columns(path)
    if "grid_id" in required:
        grid_column(path)
        available |= {"grid_id"}
    missing = sorted(set(required)-available)
    if missing: raise ValueError(f"{path}: missing columns {missing}")

def register_fit(model_id, fit, group="", specification="main", effect="irr"):
    coefficient_frames[model_id] = coefficient_table(fit, effect).reset_index().assign(
        model=model_id, family=fit["family"], group=group, specification=specification)
    matrices = []
    for label, key in [("cell", "covariance"), ("block", "block_covariance")]:
        if key not in fit: continue
        matrix = np.asarray(fit[key])
        i, j = np.indices(matrix.shape)
        matrices.append(pd.DataFrame(dict(model=model_id, covariance=label,
            term_i=np.asarray(fit["names"])[i.ravel()], term_j=np.asarray(fit["names"])[j.ravel()],
            value=matrix.ravel())))
    covariance_frames[model_id] = pd.concat(matrices, ignore_index=True)
    fit_summaries[model_id] = dict(model=model_id, group=group, specification=specification,
        family=fit["family"], check="fit", status="estimated", n_observations=fit["n_observations"],
        n_clusters=fit["n_clusters"], log_likelihood=fit["log_likelihood"], aic=fit["aic"],
        iterations=fit["iterations"], newton_step_se=fit["newton_step_se"], detail=json.dumps(fit["warnings"]))


def record_tests(key, table, specification="main"):
    hypothesis_frames[key] = table.assign(model=key, specification=specification).copy()


def record_plot(key, table, plot_type):
    frame = table.copy()
    for name in frame:
        if isinstance(frame[name].dtype, pd.CategoricalDtype) or isinstance(frame[name].dtype, pd.IntervalDtype):
            frame[name] = frame[name].astype(str)
    plot_frames[key] = frame.assign(plot_type=plot_type, source_model=key)


def save_results(status=None):
    if status is not None: manifest["status"] = status
    def combined(frames, defaults):
        return pd.concat(list(frames.values()), ignore_index=True) if frames else pd.DataFrame(columns=defaults)
    diagnostics = pd.concat([pd.DataFrame(diagnostic_rows), pd.DataFrame(list(fit_summaries.values())),
        *coverage_frames.values()], ignore_index=True)
    outputs = {
        "koeffizienten.csv": combined(coefficient_frames, ["model", "term", "coefficient"]),
        "hypothesentests.csv": combined(hypothesis_frames, ["model", "hypothesis_id", "coverage"]),
        "diagnostik.csv": diagnostics,
        "stichprobe.csv": pd.DataFrame(sample_rows),
        "plotdaten.parquet": combined(plot_frames, ["plot_type", "source_model"]),
        "kovarianzen.parquet": combined(covariance_frames, ["model", "covariance", "term_i", "term_j", "value"]),
    }
    for filename, table in outputs.items():
        target = BUNDLE_DIR / filename
        temporary = target.with_name(target.name+".tmp")
        if target.suffix == ".csv": table.to_csv(temporary, index=False)
        else: table.to_parquet(temporary, index=False)
        temporary.replace(target)
    manifest["outputs"] = [*outputs, "laufinfo.json"]
    manifest["updated_at"] = datetime.now(timezone.utc).isoformat()
    temporary = BUNDLE_DIR / "laufinfo.json.tmp"
    temporary.write_text(json.dumps(manifest, indent=2, ensure_ascii=False,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)), encoding="utf-8")
    temporary.replace(BUNDLE_DIR / "laufinfo.json")
    print(f"Saved {len(manifest['outputs'])} files; status={manifest['status']}: {BUNDLE_DIR}", flush=True)


def drop_model(model_id):
    for frames in [coefficient_frames, hypothesis_frames, covariance_frames, fit_summaries]:
        frames.pop(model_id, None)


In [ ]:
FOUNDING_TERMS = ["log_own_firms", "log_own_pop", "log_pop_access_ring_0_15", "log_firms_relative_car_ring_0_15",
                 "log_walk_pop_ring_0_10", "log_firms_relative_walk_ring_0_10", "population_growth_yoy",
                 "log_tt_motorway_exit", "walk_pt_routes_10min", "log_walk_pt_departures_10min", "pt_ohne_haltestelle"]
SURVIVAL_TERMS = ["log_own_pop", "log_own_same", "log_own_other", "log_pop_ring_0_15", "log_same_relative_ring_0_15",
                  "log_other_relative_ring_0_15", "log_tt_motorway_exit", "walk_pt_routes_10min", "log_walk_pt_departures_10min", "pt_ohne_haltestelle", "calendar_year"]


PREPARED_DIR = DATA_DIR / "modelldaten"
meta_path = PREPARED_DIR / "model_data_info.json"
if not meta_path.is_file(): raise FileNotFoundError("Run notebook 08 first.")
prepared = json.loads(meta_path.read_text(encoding="utf-8"))
if prepared.get("status") != "completed" or prepared.get("version") != "model-tables-3" or prepared.get("resolution_m") != CELL_SIZE:
    raise ValueError("Prepared-data profile mismatch; run notebook 08 again.")
if prepared.get("founding_terms") != FOUNDING_TERMS or prepared.get("survival_terms") != SURVIVAL_TERMS:
    raise ValueError("Prepared main-model terms differ; run notebook 08 again.")
founding = pd.read_parquet(PREPARED_DIR / prepared["tables"]["founding"]["file"])
survival_frame = pd.read_parquet(PREPARED_DIR / prepared["tables"]["survival"]["file"])
for name, frame, keys in [("founding", founding, ["grid_id", "period"]),
                           ("survival", survival_frame, ["standort_id", "period"])]:
    if len(frame) != prepared["tables"][name]["rows"] or frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError(f"Invalid prepared {name} rows; rerun 08.")
sample_rows.extend(prepared["sample_description"])
diagnostic_rows.extend(prepared["diagnostics"])
manifest.update(sources=prepared["source_files"], event_definition=prepared["event_definition"])
display(pd.DataFrame(sample_rows))

grid = pd.read_parquet(RASTER_PATH, columns=[grid_column(RASTER_PATH), "centroid_x", "centroid_y"])
grid = grid.rename(columns={"grid_id_100m": "grid_id"})
if grid.grid_id.isna().any() or grid.grid_id.duplicated().any():
    raise ValueError("Raster coordinates must identify cells uniquely.")
if not np.isfinite(grid[["centroid_x", "centroid_y"]]).all().all():
    raise ValueError("Missing projected raster coordinates.")
block_x = np.floor(grid.centroid_x / CONFIG["spatial_block_m"]).astype(int)
block_y = np.floor(grid.centroid_y / CONFIG["spatial_block_m"]).astype(int)
grid["spatial_block"] = block_x.astype(str) + "_" + block_y.astype(str)
block_map = grid.set_index("grid_id").spatial_block
for frame in [founding, survival_frame]:
    frame["spatial_block"] = frame.grid_id.map(block_map)
    if frame.spatial_block.isna().any():
        raise ValueError("A model cell has no spatial-block assignment.")

## 2. Gründungsmodell

Absolute counts, no offset, full year-quarter effects. The population effect is
mass minus relative firms, with the full covariance. Cell clusters allow temporal
dependence; 5 km blocks are an additional spatial-uncertainty sensitivity.

H4 adds log1p of average weekday departures summed over stops within ten walking
minutes. This counts departures at stops, not distinct vehicle trips. H4 is operationalized by the frequency,
route and no-stop contrasts and their joint test.

In [ ]:
def design_matrix(frame, terms, periods=False):
    X = frame[terms].astype(float)
    if periods:
        X = pd.concat([X, pd.get_dummies(frame.period, prefix="period", drop_first=True, dtype=float)], axis=1)
        X.insert(0, "const", 1.0)
    if not np.isfinite(X).all().all():
        raise ValueError("Non-finite model matrix.")
    constants = [c for c in X if c != "const" and X[c].nunique() < 2]
    if constants:
        print("Unidentified constant terms removed:", constants)
    X = X.drop(columns=constants)
    scale = X.std().replace(0, 1)
    if "const" in scale:
        scale["const"] = 1
    scaled = X / scale
    if np.linalg.matrix_rank(scaled.to_numpy().T @ scaled.to_numpy(), hermitian=True) < X.shape[1]:
        raise ValueError("Resolve exact collinearity before fitting.")
    return X, scaled, scale, constants


def check_fit_arrays(params, covariance, loglike):
    if not np.isfinite(params).all() or not np.isfinite(covariance).all() or not np.isfinite(loglike):
        raise RuntimeError("Non-finite model result.")
    if (np.diag(covariance) <= 0).any():
        raise RuntimeError("Non-positive coefficient uncertainty.")
    if np.linalg.eigvalsh(covariance).min() < -1e-8 * max(1, np.diag(covariance).max()):
        raise RuntimeError("Invalid covariance matrix.")


def numerical_check(model, params):
    information = -model.hessian(params)
    np.linalg.cholesky(information)
    bread = np.linalg.inv(information)
    if (np.diag(bread) <= 0).any():
        raise RuntimeError("Non-positive information matrix.")
    step = bread @ model.score(params)
    step_se = float(np.max(np.abs(step) / np.sqrt(np.diag(bread))))
    if not np.isfinite(step_se) or step_se > 0.05:
        raise RuntimeError(f"Premature optimizer stop: Newton correction = {step_se:.3g} standard errors.")
    return bread, step_se


def count_estimate(family, X, y, groups, scale, poisson=None, start=None):
    started = perf_counter()
    progress = {"iteration": 0, "last": started}
    print(f"{family}: {len(y):,} rows; fitting", flush=True)
    def report_nb2(_):
        progress["iteration"] += 1
        now = perf_counter()
        if progress["iteration"] == 1 or progress["iteration"] % 25 == 0 or now-progress["last"] >= 30:
            print(f"nb2: iteration={progress['iteration']}; elapsed={now-started:.0f}s", flush=True)
            progress["last"] = now
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        if family == "poisson":
            result = sm.GLM(y, X, family=sm.families.Poisson()).fit(
                maxiter=CONFIG["maxiter_poisson"], tol=1e-10,
                cov_type="cluster", cov_kwds={"groups": groups})
            converged = result.converged
            iterations = result.fit_history.get("iteration", 0)
        else:
            if start is None:
                start = np.r_[np.asarray(poisson["params"]) * scale.to_numpy(), .1]
            result = NegativeBinomial(y, X, loglike_method="nb2").fit(
                start_params=start, method="lbfgs", maxiter=CONFIG["maxiter_nb2"],
                pgtol=1e-10, factr=100, disp=0, callback=report_nb2,
                bounds=[(None, None)] * X.shape[1] +
                       [(np.log(CONFIG["alpha_lower"]), np.log(CONFIG["alpha_upper"]))],
                cov_type="cluster", cov_kwds={"groups": groups})
            converged = result.mle_retvals.get("converged", False)
            iterations = result.mle_retvals.get("iterations", 0)
        if not converged:
            raise RuntimeError(f"{family} did not converge.")
        bread, step_se = numerical_check(result.model, np.asarray(result.params))
    factor = np.r_[scale, 1.] if family == "nb2" else np.asarray(scale)
    fit = {
        "family": family, "names": [*X.columns, "alpha"] if family == "nb2" else list(X.columns),
        "params": (np.asarray(result.params) / factor).tolist(),
        "covariance": (np.asarray(result.cov_params()) / np.outer(factor, factor)).tolist(),
        "naive_covariance": (bread / np.outer(factor, factor)).tolist(),
        "log_likelihood": float(result.llf), "aic": float(result.aic),
        "converged": True, "iterations": int(iterations), "newton_step_se": step_se,
        "warnings": sorted({str(w.message) for w in caught}), "loaded_from_cache": False,
        "n_observations": len(y), "n_clusters": pd.Series(groups).nunique(),
        "covariance_method": "grid-cell cluster sandwich (statsmodels)",
    }
    check_fit_arrays(np.asarray(fit["params"]), np.asarray(fit["covariance"]), fit["log_likelihood"])
    if family == "nb2" and poisson is not None and result.llf + .01 < poisson["log_likelihood"]:
        raise RuntimeError("NB2 likelihood below Poisson; inspect optimization.")
    for message in fit["warnings"]:
        print(f"{family} warning: {message}")
    print(f"{family}: iterations={iterations}, maximum Newton correction/SE={step_se:.3g}")
    return fit


def coefficient_table(fit, effect="irr"):
    params = np.asarray(fit["params"])
    covariance = np.asarray(fit["covariance"])
    se = np.sqrt(np.diag(covariance))
    critical = stats.norm.ppf(.975)
    z = params/se
    table = pd.DataFrame({"coefficient": params, "std_error": se, "z": z, "p_value": 2*stats.norm.sf(np.abs(z)),
                          "ci_lower": params-critical*se, "ci_upper": params+critical*se}, index=fit["names"])
    table.index.name = "term"
    with np.errstate(over="ignore"):
        table[effect] = np.exp(table.coefficient)
        prefix = "irr" if effect == "irr" else "hr"
        table[prefix+"_ci_lower"] = np.exp(table.ci_lower)
        table[prefix+"_ci_upper"] = np.exp(table.ci_upper)
    if "alpha" in table.index: table.loc["alpha", [effect, prefix+"_ci_lower", prefix+"_ci_upper"]] = np.nan
    return table

def linear_effect(fit, weights):
    names = fit["names"]
    unknown = set(weights)-set(names)
    if unknown: raise ValueError(f"Contrast refers to missing/unidentified terms: {sorted(unknown)}")
    vector = np.asarray([weights.get(name, 0.) for name in names], dtype=float)
    estimate = float(vector @ np.asarray(fit["params"]))
    variance = float(vector @ np.asarray(fit["covariance"]) @ vector)
    if not np.isfinite(vector).all() or not np.isfinite(variance) or variance <= 0: raise ValueError("Contrast has non-positive variance.")
    se = float(np.sqrt(variance))
    return estimate, se

In [ ]:
POPULATION = {"log_pop_access_ring_0_15": 1, "log_firms_relative_car_ring_0_15": -1}
COUNT_TESTS = {
    "H1": (POPULATION, "positive", "tested"),
    "H2a": ({"log_firms_relative_car_ring_0_15": 1}, "positive", "tested"),
    "H4_routes": ({"walk_pt_routes_10min": 1}, "positive", "tested"),
    "H4_frequency": ({"log_walk_pt_departures_10min": 1}, "positive", "tested"),
    "H4_no_stop": ({"pt_ohne_haltestelle": 1}, "negative", "tested"),
    "H5": ({"log_tt_motorway_exit": 1}, "negative", "partial"),
}
SURVIVAL_TESTS = {
    "H2b": ({"log_same_relative_ring_0_15": 1}, "positive", "tested"),
    "H2c": ({"log_other_relative_ring_0_15": 1}, "negative", "tested"),
}


def contrast_row(identifier, estimate, variance, expected, coverage, covariance):
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError(f"{identifier}: non-positive contrast variance.")
    se = float(np.sqrt(variance))
    return dict(hypothesis_id=identifier, estimate=float(estimate), std_error=se,
                ci_lower=float(estimate - 1.96 * se), ci_upper=float(estimate + 1.96 * se),
                p_value=float(2 * stats.norm.sf(abs(estimate / se))),
                expected_direction=expected,
                direction_matches=bool(estimate > 0 if expected == "positive" else estimate < 0),
                coverage=coverage, covariance=covariance)


def hypothesis_table(fit, definitions):
    rows = []
    for identifier, (weights, expected, coverage) in definitions.items():
        if set(weights) - set(fit["names"]):
            rows.append(dict(hypothesis_id=identifier, coverage="unidentified"))
            continue
        estimate, se = linear_effect(fit, weights)
        rows.append(contrast_row(identifier, estimate, se**2, expected, coverage, "cell"))
    return pd.DataFrame(rows)

def wald_row(identifier, estimates, covariance, contrast, coverage, label):
    difference = contrast @ np.asarray(estimates)
    variance = contrast @ np.asarray(covariance) @ contrast.T
    rank = np.linalg.matrix_rank(variance)
    if rank != len(difference):
        return dict(hypothesis_id=identifier, coverage="unidentified", covariance=label,
                    detail="Singular covariance of the requested joint contrast.")
    statistic = float(difference @ np.linalg.solve(variance, difference))
    return dict(hypothesis_id=identifier, statistic=statistic, df=rank,
                p_value=float(stats.chi2.sf(statistic, rank)), coverage=coverage, covariance=label)


H4_TERMS = ["walk_pt_routes_10min", "log_walk_pt_departures_10min", "pt_ohne_haltestelle"]

def h4_joint(fit):
    if set(H4_TERMS)-set(fit["names"]):
        return dict(hypothesis_id="H4_joint", coverage="unidentified")
    positions = [fit["names"].index(t) for t in H4_TERMS]
    covariance = np.asarray(fit["covariance"])[np.ix_(positions, positions)]
    return wald_row("H4_joint", np.asarray(fit["params"])[positions], covariance,
                    np.eye(len(positions)), "tested", "cell")


In [ ]:
X_founding, X_nb_scaled, nb_scale, nb_constants = design_matrix(founding, FOUNDING_TERMS, periods=True)
y_founding = founding.births.to_numpy(dtype=float)
poisson_fit = count_estimate("poisson", X_nb_scaled, y_founding, founding.grid_id, nb_scale)
founding_fit = count_estimate("nb2", X_nb_scaled, y_founding, founding.grid_id, nb_scale, poisson_fit)
founding_coefficients = coefficient_table(founding_fit)
mu_poisson = np.exp(X_founding.to_numpy() @ np.asarray(poisson_fit["params"]))
mu_founding = np.exp(X_founding.to_numpy() @ np.asarray(founding_fit["params"])[:-1])
alpha = founding_fit["params"][-1]
pearson = (y_founding - mu_founding) / np.sqrt(mu_founding + alpha * mu_founding**2)
founding_hypotheses = pd.concat([hypothesis_table(founding_fit, COUNT_TESTS),
    pd.DataFrame([h4_joint(founding_fit)])], ignore_index=True)
display(founding_hypotheses)
display(founding_coefficients.loc[[x for x in founding_coefficients.index if not x.startswith("period_")]])
register_fit("founding_nb2", founding_fit)
register_fit("founding_poisson", poisson_fit, specification="reference")
record_tests("founding_nb2", founding_hypotheses)
save_results("founding_main_completed")

positive_frequency = np.expm1(founding.loc[founding.log_walk_pt_departures_10min.gt(0), "log_walk_pt_departures_10min"])
manifest["h4_frequency_support"] = dict(minimum=float(positive_frequency.min()), maximum=float(positive_frequency.max())) if len(positive_frequency) else None


## 3. Count-model checks

In [ ]:
comparison_rows = []
for fit, mu in [(poisson_fit, mu_poisson), (founding_fit, mu_founding)]:
    a = fit["params"][-1] if fit["family"] == "nb2" else 0
    zero = np.exp(-np.log1p(a*mu)/a) if a > 0 else np.exp(-mu)
    comparison_rows.append(dict(model=fit["family"], n_observations=len(founding),
        log_likelihood=fit["log_likelihood"], aic=fit["aic"], alpha=a,
        pearson_dispersion=float(np.sum((y_founding-mu)**2/(mu+a*mu**2))/(len(mu)-len(fit["params"]))),
        zero_observed=float(np.mean(y_founding == 0)), zero_expected=float(zero.mean())))
model_comparison = pd.DataFrame(comparison_rows)
residual_frame = founding[["grid_id", "year", "quarter", "period"]].assign(pearson=pearson, mu=mu_founding)
residual_frame["mu_bin"] = pd.qcut(mu_founding, 10, duplicates="drop")
residual_bins = residual_frame.groupby("mu_bin", observed=True).agg(
    rows=("mu", "size"), predicted_mean=("mu", "mean"), mean_residual=("pearson", "mean"),
    residual_variance=("pearson", "var")).reset_index()
adjacent = (residual_frame.year*4 + residual_frame.quarter).groupby(residual_frame.grid_id).diff().eq(1)
lag = residual_frame.groupby("grid_id").pearson.shift()
note("founding", "temporal_residual_correlation", "DESCRIPTIVE",
     str(residual_frame.loc[adjacent, "pearson"].corr(lag.loc[adjacent])))
vif = pd.Series(np.diag(np.linalg.inv(X_founding.drop(columns="const").corr())),
                index=X_founding.columns.drop("const"), name="VIF")
coverage_frames["vif_founding"] = vif.rename_axis("term").reset_index().rename(columns={"VIF":"value"}).assign(
    model="founding_nb2", check="VIF", status="diagnostic")
note("founding", "collinearity", "REVIEW" if vif.max() > 10 else "OK", f"maximum VIF={vif.max():.3g}")
note("founding", "extreme_residuals", "REVIEW", f"maximum |Pearson|={np.abs(pearson).max():.3g}; inspect binned fit")
inspection = founding[["grid_id","municipality_id","period","births"]].assign(
    prediction=mu_founding, pearson=pearson)
inspection = inspection.loc[inspection.pearson.abs().nlargest(30).index]
inspection.to_csv(PREPARED_DIR / "largest_residuals_local_only.csv", index=False)
display(model_comparison)
display(residual_bins)

stock_sensitivity = pd.DataFrame()
try:
    # One targeted functional-form check, prompted by the real run's stock-curvature flag.
    stock_frame = founding[[*FOUNDING_TERMS,"period"]].copy()
    stock_frame["stock_squared"] = (stock_frame.log_own_firms - stock_frame.log_own_firms.mean())**2
    X_stock, X_stock_scaled, stock_scale, _ = design_matrix(stock_frame, [*FOUNDING_TERMS, "stock_squared"], periods=True)
    start = pd.Series(founding_fit["params"], index=founding_fit["names"]).reindex([*X_stock.columns, "alpha"], fill_value=0)
    start = start.to_numpy() * np.r_[stock_scale, 1.]
    stock_fit = count_estimate("nb2", X_stock_scaled, y_founding, founding.grid_id, stock_scale,
                              poisson=poisson_fit, start=start)
    note("founding", "stock_sensitivity_fit", "REVIEW" if stock_fit["warnings"] else "OK",
         f"iterations={stock_fit['iterations']}; Newton correction/SE={stock_fit['newton_step_se']:.3g}; warnings={stock_fit['warnings']}")
    stock_sensitivity = hypothesis_table(stock_fit, COUNT_TESTS)
    stock_sensitivity["model"] = "NB2 with quadratic own-cell stock"
    stock_coefficient = coefficient_table(stock_fit).loc["stock_squared"]
    note("founding", "stock_curvature", "REVIEW" if stock_coefficient.p_value < .05 else "DESCRIPTIVE",
         f"quadratic stock beta={stock_coefficient.coefficient:.4g}; conditional cell-cluster p={stock_coefficient.p_value:.4g}")
    display(stock_sensitivity)
    register_fit("founding_stock_squared", stock_fit, specification="stock_squared")
    record_tests("founding_stock_squared", stock_sensitivity, "stock_squared")
    record_plot("founding_residual_bins", residual_bins, "residual_bins")
    coverage_frames["count_comparison"] = model_comparison.assign(check="count_comparison", status="diagnostic")
except (RuntimeError, ValueError, np.linalg.LinAlgError) as error:
    note("founding_stock_squared", "sensitivity_fit", "FAILED", str(error))
finally:
    for variable in ["stock_frame", "X_stock", "X_stock_scaled", "stock_fit", "start"]:
        globals().pop(variable, None)
# Main-model diagnostics are retained even if the optional curvature fit failed.
record_plot("founding_residual_bins", residual_bins, "residual_bins")
coverage_frames["count_comparison"] = model_comparison.assign(check="count_comparison", status="diagnostic")


In [ ]:
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from statsmodels.stats.multitest import multipletests

moran_rows = []
rng = np.random.default_rng(42)
for period, group in residual_frame.groupby("period", sort=True):
    if period not in ["2016Q4", "2020Q2", "2020Q4", "2023Q3", "2025Q4"]:
        continue
    data = group[["grid_id", "pearson"]].merge(grid, on="grid_id", validate="one_to_one")
    xy = data[["centroid_x", "centroid_y"]].to_numpy()
    pairs = cKDTree(xy).query_pairs(CELL_SIZE*np.sqrt(2)+.01, output_type="ndarray")
    n = len(data)
    if n < 4 or len(pairs) == 0:
        moran_rows.append(dict(period=period, n=n, islands=n, I=np.nan, p_perm=np.nan))
        continue
    W = csr_matrix((np.ones(2*len(pairs)), (np.r_[pairs[:,0], pairs[:,1]],
                    np.r_[pairs[:,1], pairs[:,0]])), shape=(n,n))
    degree = np.asarray(W.sum(axis=1)).ravel()
    W = W.multiply((1/np.maximum(degree,1))[:,None]).tocsr()
    z = data.pearson.to_numpy() - data.pearson.mean()
    denominator = z @ z
    if denominator <= 0:
        moran_rows.append(dict(period=period, n=n, islands=int((degree==0).sum()), I=np.nan, p_perm=np.nan))
        continue
    factor = n / W.sum()
    observed = float(factor*(z @ (W @ z))/denominator)
    exceed = 0
    for _ in range(999):
        shuffled = rng.permutation(z)
        simulated = float(factor*(shuffled @ (W @ shuffled))/denominator)
        exceed += abs(simulated+1/(n-1)) >= abs(observed+1/(n-1))
    moran_rows.append(dict(period=period, n=n, islands=int((degree==0).sum()),
                           I=observed, p_perm=(exceed+1)/1000))
moran_table = pd.DataFrame(moran_rows)
valid = moran_table.p_perm.notna()
moran_table["p_holm"] = np.nan
if valid.any():
    moran_table.loc[valid, "p_holm"] = multipletests(moran_table.loc[valid, "p_perm"], method="holm")[1]
note("founding", "spatial_residual_dependence", "REVIEW" if moran_table.p_holm.lt(.05).any() else "SCREEN ONLY",
     "Queen neighbours; five follow-up quarters including previous flags; 999 permutations; Holm. Screen only.")
display(moran_table[["period", "I", "p_holm"]])
record_plot("founding_moran", moran_table, "moran")


## 4. H3: local demand dependence

Preserve the supplied A/B/C classification. B is outside the prespecified A–C contrast.
Unknown classifications exclude the entire paired observation. The cross-model
covariance is calculated in identical clusters. Both groups use Poisson QMLE with
clustered uncertainty: the rare group outcomes do not require fitting an additional
dispersion parameter. This choice follows the numerical/dispersion diagnostics,
independently of the observed direction of the H3 contrast.

In [ ]:
H3_CLASSIFICATION = {
    "Gewerbe und Handwerk": {
        "A": ["118", "120", "124", "125A", "125B", "127", "128"],
        "B": ["101", *map(str, range(103, 109)), *map(str, range(110, 113)), *map(str, range(114, 118)), "119", "121", "122", "123", "126"],
        "C": ["113", "129"]},
    "Industrie": {"A": [], "B": ["209", "213"],
        "C": [*map(str, range(201, 208)), "210", "211", "212", "215", "216", "217", "218"]},
    "Handel": {"A": ["301", "302", "303", "308", "310", "312", "317"],
        "B": ["304A", "306", "309", "313", "315", "316", "320"],
        "C": ["304B", "305", "307", "311", "314", "318"]},
    "Bank und Versicherung": {"A": [], "B": list(map(str, range(401, 407))), "C": []},
    "Transport und Verkehr": {"A": ["505", "507", "508"], "B": ["502", "503", "506"], "C": ["501", "504"]},
    "Tourismus und Freizeitwirtschaft": {"A": ["601", "603", "605", "606"], "B": ["602", "604"], "C": []},
    "Information und Consulting": {"A": [], "B": ["701", "702", "703", "704", "707", "709"], "C": ["705", "706", "708", "710"]},
}

H3_CLASSIFICATION_ROWS = [{"Fachgruppe_ID": code, "sparte": sector, "demand_group": group}
    for sector, groups in H3_CLASSIFICATION.items() for group, codes in groups.items() for code in codes]

H3_GROUP_MAP = {r["Fachgruppe_ID"]: r["demand_group"] for r in H3_CLASSIFICATION_ROWS}

def normalize_fachgruppe(values):
    # Numeric parquet encodings such as 118.0 normalize to 118; suffixes remain meaningful.
    return values.astype("string").str.strip().str.upper().str.replace(r"\s+", "", regex=True).str.replace(r"^(\d{3})\.0+$", r"\1", regex=True)

def prepare_h3_counts():
    """Align sparse group births with the full existing model support, including genuine zero counts."""
    if H3_BIRTHS_PATH.is_file():
        require_columns(H3_BIRTHS_PATH, ["grid_id", "period", "Fachgruppe_ID", "births"])
        key = grid_column(H3_BIRTHS_PATH)
        sparse = pd.read_parquet(H3_BIRTHS_PATH, columns=[key, "period", "Fachgruppe_ID", "births"]).rename(columns={key: "grid_id"})
        if sparse.duplicated(["grid_id", "period", "Fachgruppe_ID"]).any():
            raise ValueError("Duplicate cell-quarter-Fachgruppe birth rows.")
        source_path = H3_BIRTHS_PATH
    else:
        # Legacy fallback uses existing assignments, not newly generated routing/datasets.
        require_columns(FIRMS_PATH, ["firm_id", "grid_id", "founding_date", "Fachgruppe_ID"])
        key = grid_column(FIRMS_PATH)
        firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", key, "founding_date", "Fachgruppe_ID"]).rename(columns={key: "grid_id"})
        if firms.firm_id.isna().any() or firms.firm_id.duplicated().any(): raise ValueError("Invalid firm IDs for H3 birth aggregation.")
        dates = pd.to_datetime(firms.founding_date, errors="raise")
        firms["period"] = dates.dt.to_period("Q").astype("string")
        sparse = firms.loc[firms.grid_id.notna() & dates.notna()].groupby(["grid_id", "period", "Fachgruppe_ID"], dropna=False).size().rename("births").reset_index()
        source_path = FIRMS_PATH
    if sparse[["grid_id", "period"]].isna().any().any(): raise ValueError("Missing H3 cell/time identifiers.")
    y = sparse.births.to_numpy(dtype=float)
    if not np.isfinite(y).all() or (y<0).any() or (y != np.floor(y)).any(): raise ValueError("Invalid Fachgruppe founding counts.")
    periods = pd.PeriodIndex(sparse.period, freq="Q")
    if not sparse.period.astype(str).eq(periods.astype(str)).all(): raise ValueError("Noncanonical H3 quarter identifiers.")
    sparse = sparse.loc[(periods.year >= START_YEAR) & (periods.year <= END_YEAR)].copy()
    sparse["Fachgruppe_ID"] = normalize_fachgruppe(sparse.Fachgruppe_ID)
    sparse["demand_group"] = sparse.Fachgruppe_ID.map(H3_GROUP_MAP).fillna("UNCLASSIFIED")
    keys = pd.MultiIndex.from_frame(founding[["grid_id", "period"]])
    aligned = sparse.groupby(["grid_id", "period", "demand_group"]).births.sum().unstack("demand_group", fill_value=0)
    aligned = aligned.reindex(index=keys, columns=["A", "B", "C", "UNCLASSIFIED"], fill_value=0).fillna(0)
    total = founding.births.to_numpy(dtype=float)
    unrecorded = total-aligned.sum(axis=1).to_numpy()
    if (unrecorded < -1e-8).any(): raise ValueError("Fachgruppe births exceed total founding counts on the model support.")
    aligned["UNCLASSIFIED"] += np.maximum(unrecorded, 0)
    # Unknown classifications could belong to A/C. Exclude the entire paired observation instead of inventing a group.
    keep = aligned.UNCLASSIFIED.eq(0).to_numpy()
    audit = sparse.merge(founding[["grid_id", "period"]], on=["grid_id", "period"], how="inner", validate="many_to_one")
    coverage = audit.groupby(["Fachgruppe_ID", "demand_group"], dropna=False).births.sum().reset_index()
    missing_codes = int(np.sum(np.maximum(unrecorded, 0)))
    if missing_codes:
        coverage = pd.concat([coverage, pd.DataFrame([{"Fachgruppe_ID": "<missing from source>", "demand_group": "UNCLASSIFIED", "births": missing_codes}])], ignore_index=True)
    report = {"source": str(source_path.relative_to(PROJECT_DIR)), 
        "classification": H3_CLASSIFICATION_ROWS, "frequency": FOUNDING_FREQUENCY,
        "rows_before": len(founding), "rows_retained": int(keep.sum()), "rows_excluded_unknown": int((~keep).sum()),
        "births_before": int(total.sum()), "births_excluded_unknown_observations": int(total[~keep].sum()),
        "unclassified_births": int(aligned.UNCLASSIFIED.sum()),
        "retained_periods": sorted(founding.loc[keep, "period"].unique().tolist()),
        "counts": {g: int(aligned.loc[keep, g].sum()) for g in ["A", "B", "C"]}}
    return aligned, keep, coverage, report

def h3_directional_scores(X, y, fit, scale, weights):
    """Per-observation influence of one population contrast; no large n-by-p score matrix."""
    from scipy.special import digamma
    w = np.asarray([weights.get(name, 0.) for name in fit["names"]])
    factor = np.r_[scale.to_numpy(), 1.] if fit["family"] == "nb2" else scale.to_numpy()
    # H_scaled^{-1} (w/scale) = scale * V_naive_original * w.
    direction = factor * (np.asarray(fit["naive_covariance"]) @ w)
    parameters = np.asarray(fit["params"])*factor
    alpha = float(parameters[-1]) if fit["family"] == "nb2" else 0.
    influence = np.empty(len(y))
    for first in range(0, len(y), 50000):
        last = min(first+50000, len(y))
        x = X.iloc[first:last].to_numpy()
        observed = np.asarray(y[first:last])
        beta = parameters[:-1] if alpha>0 else parameters
        mu = np.exp(x @ beta)
        influence[first:last] = (x @ direction[:X.shape[1]])*(observed-mu)/(1+alpha*mu)
        if alpha>0:
            size = 1/alpha
            alpha_score = -(digamma(observed+size)-digamma(size)+np.log(size)-np.log(size+mu)-(observed-mu)/(size+mu))/alpha**2
            influence[first:last] += direction[-1]*alpha_score
    if not np.isfinite(influence).all(): raise RuntimeError("Nonfinite H3 influence contributions.")
    return influence

In [ ]:
def cluster_influence(X, y, fit, scale, weights, groups):
    influence = h3_directional_scores(X, y, fit, scale, weights)
    codes, labels = pd.factorize(np.asarray(groups), sort=False)
    if (codes < 0).any() or len(labels) < 2:
        raise ValueError("At least two complete clusters are required.")
    n, p, g = len(y), len(fit["params"]), len(labels)
    adjustment = g / (g - 1) * (n - 1) / (n - p)
    return np.bincount(codes, weights=influence, minlength=g) * np.sqrt(adjustment)


# Larger blocks change uncertainty, not coefficients. Their boundaries remain an assumption.
block_rows = []
for identifier, (weights, expected, coverage) in COUNT_TESTS.items():
    if set(weights) - set(founding_fit["names"]):
        continue
    estimate, se = linear_effect(founding_fit, weights)
    cell_scores = cluster_influence(X_nb_scaled, y_founding, founding_fit, nb_scale, weights, founding.grid_id)
    np.testing.assert_allclose(cell_scores @ cell_scores, se**2, rtol=2e-4, atol=1e-9)
    block_scores = cluster_influence(X_nb_scaled, y_founding, founding_fit, nb_scale, weights, founding.spatial_block)
    block_rows.append(contrast_row(identifier, estimate, block_scores @ block_scores,
                                   expected, coverage, f"{CONFIG['spatial_block_m']}m blocks"))

if set(H4_TERMS).issubset(founding_fit["names"]):
    scores = np.column_stack([cluster_influence(X_nb_scaled, y_founding, founding_fit, nb_scale,
        {term: 1}, founding.spatial_block) for term in H4_TERMS])
    estimates = [founding_fit["params"][founding_fit["names"].index(t)] for t in H4_TERMS]
    block_rows.append(wald_row("H4_joint", estimates, scores.T @ scores, np.eye(len(H4_TERMS)),
                               "tested", f"{CONFIG['spatial_block_m']}m blocks"))
founding_block_hypotheses = pd.DataFrame(block_rows)

# H3 uses matched Poisson QMLE models: sparse groups do not need an extra alpha parameter.
aligned, keep, h3_coverage, h3_report = prepare_h3_counts()
if keep.all():
    h3_frame = founding
    X_h3, X_h3_scaled, h3_scale = X_founding, X_nb_scaled, nb_scale
else:
    h3_frame = founding.loc[keep].reset_index(drop=True)
    X_h3, X_h3_scaled, h3_scale, _ = design_matrix(h3_frame, FOUNDING_TERMS, periods=True)
h3_fits, h3_rows, h3_group_rows, h3_scores = {}, [], [], {}
for group in ["A", "C"]:
    counts = aligned.loc[keep, group].to_numpy(dtype=float)
    if counts.sum() == 0:
        raise ValueError(f"H3 group {group} has no births.")
    fit = count_estimate("poisson", X_h3_scaled, counts, h3_frame.grid_id, h3_scale)
    h3_fits[group] = fit
    register_fit("h3_"+group, fit, group=group)
    estimate, se = linear_effect(fit, POPULATION)
    h3_group_rows.append(dict(group=group, population_effect=estimate, std_error=se,
                             ci_lower=estimate-1.96*se, ci_upper=estimate+1.96*se,
                             births=int(counts.sum()), family="poisson", alpha=0.,
                             iterations=fit["iterations"], newton_step_se=fit["newton_step_se"],
                             fit_warnings=json.dumps(fit["warnings"])))
    for label, clusters in [("cell", h3_frame.grid_id), ("block", h3_frame.spatial_block)]:
        scores = cluster_influence(X_h3_scaled, counts, fit, h3_scale, POPULATION, clusters)
        h3_scores[group, label] = scores
        if label == "cell":
            np.testing.assert_allclose(scores @ scores, se**2, rtol=2e-4, atol=1e-9)
for label in ["cell", "block"]:
    difference = h3_scores["A", label] - h3_scores["C", label]
    estimate = h3_group_rows[0]["population_effect"] - h3_group_rows[1]["population_effect"]
    h3_rows.append(contrast_row("H3", estimate, difference @ difference, "positive", "tested", label))
h3_population_effects = pd.DataFrame(h3_group_rows)
h3_hypotheses = pd.DataFrame(h3_rows)
display(h3_population_effects)
display(h3_hypotheses)

record_tests("founding_nb2_blocks", founding_block_hypotheses)
record_tests("h3_A_vs_C", h3_hypotheses)
record_plot("h3_population", h3_population_effects, "h3_population")
coverage_frames["h3_coverage"] = h3_coverage.assign(model="h3", check="classification_coverage", status="coverage")
coverage_frames["h3_mapping"] = pd.DataFrame(H3_CLASSIFICATION_ROWS).assign(model="h3", check="classification", status="definition")
manifest["h3"] = {k:v for k,v in h3_report.items() if k != "classification"}


## 4b. H5 — Motorway effects by Sparte

Seven Poisson QMLE count models use official Sparte assignments, the existing controls
and quarter effects. Genuine zeros remain in the common cell-quarter sample. All slopes
may differ. The joint Wald test compares motorway slopes with covariance across shared
grid-cell clusters; larger spatial blocks provide a sensitivity. Failed or unidentified
sector fits remain visible. Individual p-values receive Holm correction across Sparten.


In [ ]:

def prepare_h5_counts():
    """Use official Sparte labels from existing assignments, including zero cell-quarters."""
    key = grid_column(FIRMS_PATH)
    firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", key, "founding_date", "Sparte_ID", "Sparte_Text"])
    firms = firms.rename(columns={key: "grid_id"})
    if firms.firm_id.isna().any() or firms.firm_id.duplicated().any():
        raise ValueError("Invalid firm IDs for H5 birth aggregation.")
    firms["sparte"] = firms.Sparte_ID.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)
    firms["sparte"] = firms.sparte.where(firms.sparte.isin(list("1234567")), "UNCLASSIFIED")
    labels = firms.loc[firms.sparte.ne("UNCLASSIFIED"), ["sparte", "Sparte_Text"]].dropna().drop_duplicates()
    if labels.sparte.duplicated().any():
        raise ValueError("A Sparte_ID has conflicting official names.")
    names = labels.set_index("sparte").Sparte_Text.to_dict()
    if set(names) != set("1234567"):
        raise ValueError("H5 requires the official labels for all seven Sparten.")
    dates = pd.to_datetime(firms.founding_date, errors="raise")
    firms["period"] = dates.dt.to_period("Q").astype("string")
    in_period = dates.ge(f"{START_YEAR}-01-01") & dates.lt(f"{END_YEAR+1}-01-01")
    sparse = firms.loc[in_period & firms.grid_id.notna()].groupby(["grid_id", "period", "sparte"]).size().unstack("sparte", fill_value=0)
    keys = pd.MultiIndex.from_frame(founding[["grid_id", "period"]])
    aligned = sparse.reindex(index=keys, columns=[*list("1234567"), "UNCLASSIFIED"], fill_value=0).fillna(0)
    unrecorded = founding.births.to_numpy()-aligned.sum(axis=1).to_numpy()
    if (unrecorded < 0).any():
        raise ValueError("Sparte births exceed panel births on the model sample.")
    aligned["UNCLASSIFIED"] += unrecorded
    keep = aligned.UNCLASSIFIED.eq(0).to_numpy()
    report = dict(source=str(FIRMS_PATH.relative_to(PROJECT_DIR)), rows_before=len(founding),
        rows_retained=int(keep.sum()), rows_excluded_unknown=int((~keep).sum()),
        unclassified_births=int(aligned.UNCLASSIFIED.sum()),
        births_excluded=int(founding.loc[~keep, "births"].sum()))
    coverage = pd.DataFrame([dict(sparte=s, sparte_name=names[s],
        births_before=int(aligned[s].sum()), births_retained=int(aligned.loc[keep, s].sum())) for s in names])
    return aligned.astype("int64"), keep, names, coverage, report


def h5_heterogeneity(estimates, influences, label, complete):
    if len(estimates) < 2:
        return dict(hypothesis_id="H5_heterogeneity", coverage="unidentified", covariance=label, p_value=np.nan,
                    detail="Fewer than two estimable sector effects.")
    scores = np.column_stack(influences)
    contrast = np.column_stack([-np.ones(len(estimates)-1), np.eye(len(estimates)-1)])
    row = wald_row("H5_heterogeneity", estimates, scores.T @ scores, contrast,
                   "tested" if complete else "partial", label)
    row["sectors_tested"] = len(estimates)
    row["detail"] = "Equality of motorway slopes; covariance includes shared grid cells."
    return row


In [ ]:

# H5 uses sector count models on matched support, not subsets of the total-count outcome.
h5_aligned, h5_keep, sector_names, h5_coverage, h5_report = prepare_h5_counts()
if h5_keep.all():
    h5_frame, X_h5_scaled, h5_scale = founding, X_nb_scaled, nb_scale
else:
    h5_frame = founding.loc[h5_keep].reset_index(drop=True)
    _, X_h5_scaled, h5_scale, _ = design_matrix(h5_frame, FOUNDING_TERMS, periods=True)
h5_rows, h5_diagnostics, h5_coefficients = [], [], []
h5_estimates, h5_scores = [], {"cell": [], "block": []}
for sector, sector_name in sorted(sector_names.items()):
    counts = h5_aligned.loc[h5_keep, sector].to_numpy(dtype=float)
    print(f"H5: {sector_name}; {int(counts.sum()):,} births", flush=True)
    try:
        if not counts.sum() or "log_tt_motorway_exit" not in X_h5_scaled:
            raise RuntimeError("No births or no identified motorway-time regressor.")
        fit = count_estimate("poisson", X_h5_scaled, counts, h5_frame.grid_id, h5_scale)
        estimate, se = linear_effect(fit, {"log_tt_motorway_exit": 1})
        sector_scores = {}
        for label, clusters in [("cell", h5_frame.grid_id), ("block", h5_frame.spatial_block)]:
            sector_scores[label] = cluster_influence(X_h5_scaled, counts, fit, h5_scale,
                                                   {"log_tt_motorway_exit": 1}, clusters)
        np.testing.assert_allclose(sector_scores["cell"] @ sector_scores["cell"], se**2, rtol=2e-4, atol=1e-9)
    except (RuntimeError, ValueError, np.linalg.LinAlgError) as error:
        h5_diagnostics.append(dict(sparte=sector, sparte_name=sector_name, status="unidentified", detail=str(error)))
        h5_rows.append(dict(hypothesis_id="H5_sector", sparte=sector, sparte_name=sector_name,
                            coverage="unidentified", covariance="cell", p_value=np.nan, detail=str(error)))
        continue
    register_fit("h5_"+sector, fit, group=sector)
    h5_estimates.append(estimate)
    h5_coefficients.append(coefficient_table(fit).reset_index().assign(sparte=sector, sparte_name=sector_name))
    h5_diagnostics.append(dict(sparte=sector, sparte_name=sector_name, status="estimated",
        births=int(counts.sum()), family="poisson_qmle", iterations=fit["iterations"],
        newton_step_se=fit["newton_step_se"], detail=json.dumps(fit["warnings"])))
    for label in ["cell", "block"]:
        scores = sector_scores[label]
        h5_scores[label].append(scores)
        row = contrast_row("H5_sector", estimate, scores @ scores, "negative", "tested", label)
        row.update(sparte=sector, sparte_name=sector_name, births=int(counts.sum()))
        h5_rows.append(row)
for label in ["cell", "block"]:
    h5_rows.append(h5_heterogeneity(h5_estimates, h5_scores[label], label, len(h5_estimates)==7))
h5_hypotheses = pd.DataFrame(h5_rows)
for label in ["cell", "block"]:
    valid = h5_hypotheses.hypothesis_id.eq("H5_sector") & h5_hypotheses.covariance.eq(label) & h5_hypotheses.p_value.notna()
    if valid.any():
        h5_hypotheses.loc[valid, "p_holm_sectors"] = multipletests(h5_hypotheses.loc[valid, "p_value"], method="holm")[1]
h5_coefficients = pd.concat(h5_coefficients, ignore_index=True) if h5_coefficients else pd.DataFrame()
h5_diagnostics = pd.DataFrame(h5_diagnostics)
display(h5_hypotheses)
del h5_aligned, h5_frame, X_h5_scaled, h5_scores


record_tests("h5_sectors", h5_hypotheses)
coverage_frames["h5_coverage"] = h5_coverage.assign(model="h5", check="sector_coverage", status="coverage")
coverage_frames["h5_diagnostics"] = h5_diagnostics.assign(model="h5", check="sector_fit")
manifest["h5"] = h5_report
save_results("founding_completed")


## 4c. Prespecified count sensitivities

Use the existing 30-minute accessibility, add urban-centre travel time, and omit the
2019/2020 population-anchor transition year. Each is a separate comparison with the
main specification. Failed fits remain diagnostic records; no specification is selected
automatically from its p-values. Missing extra covariates do not change the main sample.


In [ ]:
# Three prespecified checks; keep the main model unchanged.
def count_sensitivity(model_id, frame, terms, definitions):
    print("Count sensitivity:", model_id, flush=True)
    before = len(frame)
    frame = frame.dropna(subset=terms).reset_index(drop=True)
    sample_rows.append(dict(model=model_id, stage="model_sample", rows=len(frame),
        units=int(frame.grid_id.nunique()), events=int(frame.births.sum()), excluded=before-len(frame)))
    try:
        _, scaled, scale, _ = design_matrix(frame, terms, periods=True)
        reference = count_estimate("poisson", scaled, frame.births.to_numpy(dtype=float), frame.grid_id, scale)
        initial = pd.Series(founding_fit["params"], index=founding_fit["names"])
        keys = [t.replace("ring_0_30", "ring_0_15") for t in scaled.columns]+["alpha"]
        start = initial.reindex(keys, fill_value=0).to_numpy()*np.r_[scale, 1.]
        fit = count_estimate("nb2", scaled, frame.births.to_numpy(dtype=float), frame.grid_id, scale, reference, start=start)
        register_fit(model_id, fit, specification=model_id)
        record_tests(model_id, hypothesis_table(fit, definitions), model_id)
    except (RuntimeError, ValueError, np.linalg.LinAlgError) as error:
        note(model_id, "sensitivity_fit", "FAILED", str(error))
        print(model_id, "failed:", error, flush=True)

if CONFIG["accessibility_sensitivity_minutes"] not in [None, 30]:
    raise ValueError("The prespecified alternative is 30 minutes, or None to disable it.")
if CONFIG["accessibility_sensitivity_minutes"] == 30:
    terms30 = [t.replace("car_ring_0_15", "car_ring_0_30").replace("access_ring_0_15", "access_ring_0_30") for t in FOUNDING_TERMS]
    tests30 = {key: ({t.replace("car_ring_0_15", "car_ring_0_30").replace("access_ring_0_15", "access_ring_0_30"):v for t,v in weights.items()}, expected, coverage)
        for key,(weights,expected,coverage) in COUNT_TESTS.items()}
    count_sensitivity("founding_access_30min", founding, terms30, tests30)
if CONFIG["centre_sensitivity"]:
    count_sensitivity("founding_centre_control", founding, [*FOUNDING_TERMS, "log_tt_urban_centre"], COUNT_TESTS)
if CONFIG["population_transition_sensitivity"]:
    count_sensitivity("founding_without_2020", founding.loc[founding.year.ne(2020)], FOUNDING_TERMS, COUNT_TESTS)

# Keep only aggregate influence results before the survival fit.
del X_founding, X_nb_scaled, X_h3, X_h3_scaled, h3_frame, founding, residual_frame
del y_founding, mu_poisson, mu_founding, pearson, aligned, counts
save_results("founding_completed")


## 5. Überlebensmodell

Age since founding, delayed entry, quarterly intervals (start, stop], Efron ties.
Keep the numerically validated cluster-score calculation: the real software version's
library covariance did not pass its check. This is required statistical code.

In [ ]:
def cox_score_blocks(model, params, groups):
    # Efron case-weight influence: derivative of the weighted partial-likelihood score.
    codes, labels = pd.factorize(np.asarray(groups), sort=False)
    if (codes<0).any(): raise ValueError("Missing Cox cluster identifiers.")
    blocks = np.zeros((len(labels), model.exog.shape[1]))
    martingale = np.zeros(len(model.endog))
    used = np.zeros(len(model.endog), dtype=bool)
    event_rows, schoenfeld = [], []
    surv = model.surv
    for st in range(surv.nstrat):
        x = surv.exog_s[st]
        original = surv.stratum_rows[st]
        eta = x @ params
        if surv.offset_s is not None: eta = eta + surv.offset_s[st]
        w = np.exp(eta-eta.max())
        for k in range(len(surv.ufailt[st]))[::-1]:
            event_time = surv.ufailt[st][k]
            # Explicit membership also handles intervals spanning no event time.
            risk = np.flatnonzero((surv.entry_s[st]<=event_time) & (surv.time_s[st]>=event_time))
            death = surv.ufailt_ix[st][k]
            m = len(death)
            S0 = w[risk].sum()
            S1 = w[risk] @ x[risk]
            E0 = w[death].sum()
            E1 = w[death] @ x[death]
            if model.ties == "efron":
                fraction = np.arange(m)/m
                denominator = S0-fraction*E0
                mean = (S1[None,:]-fraction[:,None]*E1)/denominator[:,None]
                h = np.sum(1/denominator)
                hx = np.sum(mean/denominator[:,None], axis=0)
                hf = np.sum(fraction/denominator)
                hxf = np.sum(fraction[:,None]*mean/denominator[:,None], axis=0)
                average = mean.mean(axis=0)
            else:
                average = S1/S0
                h, hx, hf, hxf = m/S0, m/S0*average, 0., np.zeros(x.shape[1])
            score_risk = -w[risk,None]*(x[risk]*h-hx)
            score_death = x[death]-average + w[death,None]*(x[death]*hf-hxf)
            np.add.at(blocks, codes[original[risk]], score_risk)
            np.add.at(blocks, codes[original[death]], score_death)
            martingale[original[risk]] -= w[risk]*h
            martingale[original[death]] += 1+w[death]*hf
            used[original[risk]] = True
            event_rows.extend(original[death])
            schoenfeld.extend(x[death]-average)
    return blocks, np.asarray(event_rows, dtype=int), np.asarray(schoenfeld), martingale, used

def verify_cox_cluster_implementation():
    # Independent small weighted-score calculation: includes ties, entry and strata.
    x = np.array([[0.,1.],[1.,0.],[.5,.4],[-.3,1.2],[1.1,.2],[.2,-.4],
                  [.1,.8],[.7,.1],[-.4,.3],[1.3,.7],[.4,-.2],[.9,.5],[.1,.3],[.6,.5]])
    stop = np.array([1,2,2,3,4,4,1,2,3,3,4,5,2.,3.], dtype=float)
    entry = np.nextafter(np.array([0,0,1,0,2,3,0,0,1,2,0,4,0.,2.1], dtype=float), np.inf)
    event = np.array([0,1,1,0,1,0,1,0,1,1,0,1,0,0])
    strata = np.r_[np.zeros(6), np.ones(6), 2, 0]
    groups = np.array([0,1,1,2,3,3,4,4,5,5,6,6,7,8])
    params = np.array([.15, -.25])
    library_valid = True
    for ties in ["efron", "breslow"]:
        model = PHReg(stop, x, status=event, entry=entry, strata=strata, ties=ties)
        blocks, _, _, _, _ = cox_score_blocks(model, params, groups)
        def weighted_score(case_weights):
            total = np.zeros(2)
            for stratum in np.unique(strata):
                for t in np.unique(stop[(strata==stratum) & (event==1)]):
                    risk = (strata==stratum) & (entry<=t) & (stop>=t)
                    died = (strata==stratum) & (stop==t) & (event==1)
                    hazard = case_weights*np.exp(x@params)
                    S0, S1 = hazard[risk].sum(), hazard[risk]@x[risk]
                    E0, E1 = hazard[died].sum(), hazard[died]@x[died]
                    m = int(died.sum())
                    f = np.arange(m)/m if ties=="efron" else np.zeros(m)
                    total += case_weights[died]@x[died]
                    total -= case_weights[died].sum()/m*np.sum((S1[None,:]-f[:,None]*E1)/(S0-f*E0)[:,None], axis=0)
            return total
        numerical = []
        for group in pd.unique(groups):
            delta = groups==group
            numerical.append((weighted_score(1+1e-5*delta)-weighted_score(1-1e-5*delta))/(2e-5))
        np.testing.assert_allclose(blocks, numerical, rtol=2e-5, atol=1e-8)
        np.testing.assert_allclose(blocks.sum(axis=0), model.score(params), rtol=1e-8, atol=1e-8)
        bread = np.linalg.inv(-model.hessian(params))
        checked = bread @ (blocks.T@blocks) @ bread
        model.groups = groups
        library = model.robust_covariance(params)
        library_valid &= bool(np.isfinite(library).all() and np.allclose(library, checked, rtol=1e-6, atol=1e-8))
    print("Cox CR0: finite-difference cluster-score checks passed; library covariance validated:", library_valid)
    return library_valid

_cox_diagnostic_state = None
def make_cox_model(ties="efron"):
    return PHReg(survival_frame.stop.to_numpy(dtype=float), X_cox_scaled,
        status=survival_frame.event.to_numpy(), entry=np.nextafter(survival_frame.start.to_numpy(dtype=float), np.inf),
        strata=survival_frame.Fachgruppe_ID.to_numpy(), ties=ties)

def cox_newton(model, start=None, maxiter=None, label="Cox"):
    """Safeguarded Newton steps on the unchanged PHReg partial likelihood."""
    params = np.zeros(model.exog.shape[1]) if start is None else np.asarray(start, dtype=float).copy()
    maxiter = CONFIG["maxiter_cox"] if maxiter is None else maxiter
    if params.shape != (model.exog.shape[1],) or not np.isfinite(params).all():
        raise ValueError("Invalid Cox starting values.")
    started = perf_counter()
    evaluations = 1
    loglike = float(model.loglike(params))
    print(f"{label}: safeguarded Newton; {len(params)} terms; maxiter={maxiter}", flush=True)
    for iteration in range(maxiter+1):
        information = -model.hessian(params)
        np.linalg.cholesky(information)
        bread = np.linalg.inv(information)
        score = model.score(params)
        step = bread @ score
        step_se = float(np.max(np.abs(step)/np.sqrt(np.diag(bread))))
        print(f"{label}: iteration={iteration}; loglik={loglike:.6f}; Newton correction/SE={step_se:.3g}; "
              f"elapsed={perf_counter()-started:.0f}s", flush=True)
        if not np.isfinite(loglike) or not np.isfinite(step_se):
            raise RuntimeError(f"{label}: non-finite likelihood or Newton step.")
        if step_se <= .005:
            return dict(params=params, bread=bread, loglike=loglike, newton_step_se=step_se,
                        iterations=iteration, function_evaluations=evaluations)
        if iteration == maxiter:
            break
        # Backtracking prevents a Newton step from reducing the likelihood.
        fraction = 1.
        for _ in range(20):
            candidate = params+fraction*step
            candidate_loglike = float(model.loglike(candidate))
            evaluations += 1
            if np.isfinite(candidate_loglike) and candidate_loglike >= loglike+1e-4*fraction*float(score @ step):
                params, loglike = candidate, candidate_loglike
                break
            fraction *= .5
        else:
            raise RuntimeError(f"{label}: Newton line search failed; last correction={step_se:.3g} SE.")
    raise RuntimeError(f"{label}: no convergence after {maxiter} Newton steps; last correction={step_se:.3g} SE.")


def estimate_cox(model, library_valid, retain=True, start=None, maxiter=None, label="Cox main"):
    global _cox_diagnostic_state
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        raw = cox_newton(model, start=start, maxiter=maxiter, label=label)
        params, bread, step_se = raw["params"], raw["bread"], raw["newton_step_se"]
        print(f"{label}: converged; computing cluster covariance", flush=True)
        blocks, event_rows, sch, martingale, used = cox_score_blocks(model, params, survival_frame.grid_id)
        score_error = float(np.max(np.abs(blocks.sum(axis=0)-model.score(params))))
        if not np.allclose(blocks.sum(axis=0), model.score(params), rtol=1e-7, atol=1e-6):
            raise RuntimeError("Cox cluster-score identity failed; robust inference is unavailable.")
        covariance = bread @ (blocks.T@blocks) @ bread
        covariance_method = "validated Efron/Breslow case-weight cluster-score sandwich CR0"
        if library_valid:
            model.groups = survival_frame.grid_id.to_numpy()
            library_covariance = model.robust_covariance(params)
            if np.isfinite(library_covariance).all() and np.allclose(library_covariance, covariance, rtol=1e-5, atol=1e-7):
                covariance = library_covariance
                covariance_method = "validated statsmodels PHReg group covariance"
        check_fit_arrays(params, covariance, raw["loglike"])
    messages = sorted({str(w.message) for w in caught})
    for message in messages: print("Cox warning:", message)
    factor = cox_scale.to_numpy()
    cell_labels = pd.unique(survival_frame.grid_id)
    block_codes, block_labels = pd.factorize(pd.Series(cell_labels).map(block_map), sort=False)
    if (block_codes < 0).any() or len(block_labels) < 2:
        raise ValueError("Insufficient spatial blocks for Cox sensitivity.")
    spatial_scores = np.zeros((len(block_labels), blocks.shape[1]))
    np.add.at(spatial_scores, block_codes, blocks)
    spatial_covariance = bread @ (spatial_scores.T @ spatial_scores) @ bread
    if retain: _cox_diagnostic_state = (blocks, event_rows, sch, martingale, used)
    return {"family": "cox", "names": list(X_cox.columns), "params": (params/factor).tolist(),
        "covariance": (covariance/np.outer(factor, factor)).tolist(), "naive_covariance": (bread/np.outer(factor, factor)).tolist(),
        "block_covariance": (spatial_covariance/np.outer(factor, factor)).tolist(),
        "newton_step_se": step_se, "n_spatial_blocks": len(block_labels),
        "log_likelihood": raw["loglike"], "aic": float(-2*raw["loglike"]+2*len(params)),
        "converged": True, "optimizer": "safeguarded Newton", "iterations": raw["iterations"],
        "function_evaluations": raw["function_evaluations"], "warnings": messages,
        "n_observations": int(len(survival_frame)), "n_clusters": int(survival_frame.grid_id.nunique()),
        "n_events": int(survival_frame.event.sum()), "n_informative_intervals": int(used.sum()), "covariance_method": covariance_method,
        "score_identity_max_error": score_error, "rows_outside_event_risk_sets": int((~used).sum()),
        "ties": model.ties, "entry_convention": "(start, stop]; nextafter(start,+inf)"}

In [ ]:
X_cox, X_cox_scaled, cox_scale, cox_constants = design_matrix(survival_frame, SURVIVAL_TERMS)
library_covariance_valid = verify_cox_cluster_implementation()
cox_model = make_cox_model()
try:
    cox_fit = estimate_cox(cox_model, library_covariance_valid)
except (RuntimeError, ValueError, np.linalg.LinAlgError) as error:
    note("survival", "main_fit", "FAILED", str(error))
    save_results("failed")
    raise
cox_coefficients = coefficient_table(cox_fit, "hazard_ratio")
cox_hypotheses = hypothesis_table(cox_fit, SURVIVAL_TESTS)
block_fit = dict(cox_fit, covariance=cox_fit["block_covariance"])
cox_block_hypotheses = hypothesis_table(block_fit, SURVIVAL_TESTS).assign(covariance="spatial blocks")
display(cox_hypotheses)
display(cox_block_hypotheses)
display(cox_coefficients[["coefficient", "std_error", "hazard_ratio", "hr_ci_lower", "hr_ci_upper"]])
register_fit("survival_cox", cox_fit, effect="hazard_ratio")
record_tests("survival_cox", cox_hypotheses)
record_tests("survival_cox_blocks", cox_block_hypotheses)
cox_vif = pd.Series(np.diag(np.linalg.inv(X_cox.corr())), index=X_cox.columns, name="value")
coverage_frames["vif_survival"] = cox_vif.rename_axis("term").reset_index().assign(model="survival_cox", check="VIF", status="diagnostic")


In [ ]:
blocks, event_rows, sch, martingale, used = _cox_diagnostic_state
ph_rows, ph_bins = [], []
event_age = survival_frame.stop.to_numpy()[event_rows]/4
for j, term in enumerate(X_cox.columns):
    correlation = stats.spearmanr(np.log1p(event_age), sch[:,j])
    ph_rows.append(dict(term=term, spearman_rho=correlation.statistic, screen_p_value=correlation.pvalue))
    binned = pd.DataFrame({"age_years": event_age, "residual": sch[:,j]})
    binned["age_band"] = pd.cut(event_age, [0,2,5,10,20,np.inf])
    summary = binned.groupby("age_band", observed=True).agg(n_events=("residual","size"),
        mean_age_years=("age_years","mean"), mean_schoenfeld=("residual","mean")).reset_index()
    summary["age_band"] = summary.age_band.astype(str)
    summary["term"] = term
    ph_bins.append(summary.loc[summary.n_events.ge(5)])
ph_screen = pd.DataFrame(ph_rows)
valid = ph_screen.screen_p_value.notna()
ph_screen.loc[valid, "p_holm"] = multipletests(ph_screen.loc[valid,"screen_p_value"], method="holm")[1]
note("survival", "proportional_hazards", "REVIEW" if ph_screen.p_holm.lt(.05).any() else "SCREEN ONLY",
     "Schoenfeld/event-age correlations are a screen, not a cluster-robust formal PH test.")
display(ph_screen)

age_effects = pd.DataFrame()
manifest["age_sensitivity"] = {"status": "pending" if CONFIG["ph_sensitivity"] else "disabled"}

record_plot("survival_schoenfeld", ph_screen, "ph_screen")
record_plot("survival_schoenfeld_bins", pd.concat(ph_bins, ignore_index=True), "ph_bins")

manifest["accessibility_sensitivity"] = dict(status="pending" if CONFIG["accessibility_sensitivity_minutes"] == 30 else "disabled", minutes=CONFIG["accessibility_sensitivity_minutes"])


## 6. Export

H4 covers lines, departures and the no-stop indicator under its revised definition. H5 includes the common effect and the sector comparison. All estimates describe
associations. Spatial blocks and age sensitivity do not remove every model limitation.
Main aggregate results are saved here, before the optional age sensitivity in section 7.


In [ ]:
locations = survival_frame.groupby("standort_id").agg(
    entry=("start","min"), exit=("stop","max"), event=("event","max"))
manifest["sample_summary"] = dict(locations=len(locations), exits=int(locations.event.sum()),
    right_censored=int(locations.event.eq(0).sum()), left_truncated=int(locations.entry.gt(0).sum()))

# Relative founding counts for an interpretable H4 frequency plot, conditional on the
# other covariates and on stop availability. The reference is 10 departures/day.
term = "log_walk_pt_departures_10min"
if term in founding_fit["names"]:
    estimate, se = linear_effect(founding_fit, {term: 1})
    values = np.array([1., 2., 5., 10., 20., 50., 100., 200., 500., 1000.])
    support = manifest.get("h4_frequency_support")
    reference = 10.
    if support is not None:
        reference = float(np.clip(reference, support["minimum"], support["maximum"]))
        values = values[(values>=support["minimum"]) & (values<=support["maximum"])]
    values = np.unique(np.r_[values, reference])
    delta = np.log1p(values)-np.log1p(reference)
    effect = estimate*delta
    uncertainty = se*np.abs(delta)
    record_plot("h4_frequency", pd.DataFrame(dict(departures=values, reference=reference,
        relative_count=np.exp(effect), ci_lower=np.exp(effect-1.96*uncertainty),
        ci_upper=np.exp(effect+1.96*uncertainty))), "h4_frequency")
save_results("completed")
print("Main results are complete. Optional Cox sensitivities follow below.")


## 7. Age-varying Cox sensitivity

The main aggregate bundle has already been saved. This optional fit starts from the main
coefficients and has its own iteration limit and progress output. Its failure is recorded
without discarding the main results. The log-age interactions use each quarterly interval's
end age; they are a quarterly approximation, not an exact continuous-time PH correction.


In [ ]:
# One prespecified time-varying-coefficient sensitivity, rather than searching age cutoffs.
age_effects = []
drop_model("survival_age_varying")
if CONFIG["ph_sensitivity"]:
    base_X, base_scaled, base_scale = X_cox, X_cox_scaled, cox_scale
    base_state = _cox_diagnostic_state
    try:
        log_age = np.log1p(survival_frame.stop.to_numpy()/4)
        varying = pd.concat([base_X, base_X.mul(log_age, axis=0).add_suffix("_log_age")], axis=1)
        X_cox, X_cox_scaled, cox_scale, _ = design_matrix(varying, list(varying.columns))
        # Start at the fitted main model, with zero interaction coefficients.
        start = np.asarray([dict(zip(cox_fit["names"], cox_fit["params"])).get(t, 0.) for t in X_cox.columns])*cox_scale.to_numpy()
        age_fit = estimate_cox(make_cox_model(), library_covariance_valid, retain=False,
            start=start, maxiter=CONFIG["maxiter_cox_sensitivity"], label="Cox age sensitivity")
        register_fit("survival_age_varying", age_fit, specification="age_varying", effect="hazard_ratio")
        note("survival", "age_sensitivity_fit", "REVIEW" if age_fit["warnings"] else "OK",
             f"iterations={age_fit['iterations']}; Newton correction/SE={age_fit['newton_step_se']:.3g}; warnings={age_fit['warnings']}")
        for identifier, (weights, expected, coverage) in SURVIVAL_TESTS.items():
            for years in [1,5,10]:
                contrast = dict(weights)
                contrast.update({name+"_log_age": value*np.log1p(years) for name,value in weights.items()})
                if set(contrast)-set(age_fit["names"]):
                    continue
                for covariance in ["cell", "block"]:
                    use_fit = age_fit if covariance=="cell" else dict(age_fit, covariance=age_fit["block_covariance"])
                    estimate, se = linear_effect(use_fit, contrast)
                    row = contrast_row(identifier, estimate, se**2, expected, coverage, covariance)
                    row["age_years"] = years
                    age_effects.append(row)
        manifest["age_sensitivity"] = dict(status="completed", iterations=age_fit["iterations"],
            newton_step_se=age_fit["newton_step_se"], maxiter=CONFIG["maxiter_cox_sensitivity"])
    except (RuntimeError, ValueError, np.linalg.LinAlgError) as error:
        age_effects = []
        manifest["age_sensitivity"] = dict(status="failed", detail=str(error))
        note("survival", "age_sensitivity_fit", "FAILED", str(error))
        print("Age sensitivity failed; the completed main bundle remains available:", error, flush=True)
    finally:
        X_cox, X_cox_scaled, cox_scale = base_X, base_scaled, base_scale
        _cox_diagnostic_state = base_state
    if "varying" in globals():
        del varying
    age_effects = pd.DataFrame(age_effects)
    record_tests("survival_age_varying", age_effects, "age_varying")
    save_results("completed")
if not CONFIG["ph_sensitivity"]:
    manifest["age_sensitivity"] = dict(status="disabled")
    save_results("completed")
display(age_effects)


## 8. Thirty-minute Cox sensitivity

The main results and their diagnostic state are retained. Histories with incomplete
30-minute covariates are excluded as whole histories and reported separately.


In [ ]:
drop_model("survival_access_30min")
drop_model("survival_access_30min_blocks")
if CONFIG["accessibility_sensitivity_minutes"] == 30:
    base_frame, base_X, base_scaled, base_scale = survival_frame, X_cox, X_cox_scaled, cox_scale
    base_state = _cox_diagnostic_state
    terms30 = [t.replace("ring_0_15", "ring_0_30") for t in SURVIVAL_TERMS]
    invalid_ids = set(base_frame.loc[~np.isfinite(base_frame[terms30]).all(axis=1), "standort_id"])
    try:
        survival_frame = base_frame.loc[~base_frame.standort_id.isin(invalid_ids)].reset_index(drop=True)
        sample_rows.append(dict(model="survival_access_30min", stage="model_sample", rows=len(survival_frame),
            units=int(survival_frame.standort_id.nunique()), events=int(survival_frame.event.sum()), excluded=len(invalid_ids)))
        X_cox, X_cox_scaled, cox_scale, _ = design_matrix(survival_frame, terms30)
        start = np.asarray([dict(zip(cox_fit["names"], cox_fit["params"])).get(t.replace("ring_0_30", "ring_0_15"), 0.) for t in X_cox.columns])*cox_scale.to_numpy()
        fit30 = estimate_cox(make_cox_model(), library_covariance_valid, retain=False, start=start, maxiter=CONFIG["maxiter_cox_sensitivity"], label="Cox 30-minute sensitivity")
        register_fit("survival_access_30min", fit30, specification="access_30min", effect="hazard_ratio")
        tests30 = {key: ({t.replace("ring_0_15", "ring_0_30"):v for t,v in weights.items()}, expected, coverage)
            for key,(weights,expected,coverage) in SURVIVAL_TESTS.items()}
        record_tests("survival_access_30min", hypothesis_table(fit30, tests30), "access_30min")
        record_tests("survival_access_30min_blocks", hypothesis_table(dict(fit30, covariance=fit30["block_covariance"]), tests30).assign(covariance="block"), "access_30min")
        manifest["accessibility_sensitivity"] = dict(status="completed", minutes=30)
    except (RuntimeError, ValueError, np.linalg.LinAlgError) as error:
        manifest["accessibility_sensitivity"] = dict(status="failed", minutes=30, detail=str(error))
        note("survival_access_30min", "sensitivity_fit", "FAILED", str(error))
    finally:
        survival_frame, X_cox, X_cox_scaled, cox_scale = base_frame, base_X, base_scaled, base_scale
        _cox_diagnostic_state = base_state
    save_results("completed")

if CONFIG["accessibility_sensitivity_minutes"] is None:
    manifest["accessibility_sensitivity"] = dict(status="disabled", minutes=None)
    save_results("completed")
